# `district_code` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `district_code`.

            ## Relationships selected in advance

            - `region_code` — Region context disambiguates the reused district label.
- `lga` — LGA gives a named administrative comparison.
- `ward` — Ward is the finer administrative level.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'district_code'
feature_metadata = {'order': 14, 'name': 'district_code', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'cast to category and combine with region context', 'finding': 'The integer code is reused across regions and zero is not universally missing.', 'decision': 'Treat as categorical and prefer a region-code/district-code composite if retained.', 'risk': 'The same raw code does not identify one national district.', 'related': [{'feature': 'region_code', 'reason': 'Region context disambiguates the reused district label.'}, {'feature': 'lga', 'reason': 'LGA gives a named administrative comparison.'}, {'feature': 'ward', 'reason': 'Ward is the finer administrative level.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for district_code.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,region_code,Region context disambiguates the reused distri...
1,lga,LGA gives a named administrative comparison.
2,ward,Ward is the finer administrative level.


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,district_code,region_code,bias-corrected Cramer's V,0.4192,59400,20,27,21.46,36.13,Region context disambiguates the reused distri...
1,district_code,lga,bias-corrected Cramer's V,0.8565,59400,20,125,20.15,97.71,LGA gives a named administrative comparison.
2,district_code,ward,bias-corrected Cramer's V,0.9370,59400,20,2092,3.61,96.95,Ward is the finer administrative level.


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `district_code`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **cast to category and combine with region context**.
